# Comparing AWS deployment approaches: boto3 vs CloudFormation vs CDK

Purpose: deploy the same small stack three ways and feel the difference. The stack is one versioned S3 bucket with a couple of tags — small enough to hold in my head, real enough that the trade-offs show. This is one way to compare them; the docs also suggest starting from CloudFormation when the team wants a reviewed template artifact.

Plan: (1) sketch the imperative boto3 call sequence, (2) write the declarative CloudFormation template, (3) mirror it as a CDK construct, then verify all three describe the same resources and note when I would reach for each.

In [ ]:
# last_verified: 2026-09-23 · AWS (n/a)
# Shared fixture: the one small stack everything below must describe.
STACK = {
    "bucket_name": "example-demo-bucket",
    "versioning": "Enabled",
    "tags": {"project": "demo", "managed-by": "notebook"},
}
print("stack spec:", STACK)

## Step 1 — boto3 (imperative SDK calls)

What I tried: list the exact API calls in order, dry-run style. Nothing here touches the network — I wanted the sequence right before ever handing it credentials. The thing I keep tripping on with the SDK path is ordering: bucket first, then versioning, then tagging, and every step needs its own error handling.

In [ ]:
try:
    import boto3  # only used to show client construction; never called
    HAVE_BOTO3 = True
except ImportError:
    HAVE_BOTO3 = False

# The imperative plan: one API call per mutation, in dependency order.
boto3_plan = [
    ("create_bucket", {"Bucket": STACK["bucket_name"]}),
    ("put_bucket_versioning", {"Bucket": STACK["bucket_name"],
                                "VersioningConfiguration": {"Status": STACK["versioning"]}}),
    ("put_bucket_tagging", {"Bucket": STACK["bucket_name"],
                             "Tagging": {"TagSet": [{"Key": k, "Value": v}
                                                         for k, v in STACK["tags"].items()]}}),
]
if HAVE_BOTO3:
    client = boto3.client("s3", region_name="us-east-1")
    print("boto3 client ready:", type(client).__name__)
else:
    print("boto3 not installed here — plan below is still the full call list")
for op, kwargs in boto3_plan:
    print("s3." + op, "->", sorted(kwargs)) 
assert [op for op, _ in boto3_plan] == ["create_bucket", "put_bucket_versioning", "put_bucket_tagging"]
assert len(boto3_plan[2][1]["Tagging"]["TagSet"]) == len(STACK["tags"])
print("boto3 plan covers bucket + versioning +", len(STACK["tags"]), "tags")

## Step 2 — CloudFormation (declarative template)

Same stack, stated as desired state instead of steps. I wrote the template as JSON so this notebook can parse and check it with the standard library. What surprised me: the template says nothing about order — the service figures out that versioning and tags belong to the bucket.

In [ ]:
import json

cfn_template = {
    "AWSTemplateFormatVersion": "2010-09-09",
    "Description": "Demo stack: one versioned, tagged bucket (notebook comparison).",
    "Resources": {
        "DemoBucket": {
            "Type": "AWS::S3::Bucket",
            "Properties": {
                "BucketName": STACK["bucket_name"],
                "VersioningConfiguration": {"Status": STACK["versioning"]},
                "Tags": [{"Key": k, "Value": v} for k, v in STACK["tags"].items()],
            },
        }
    },
}
# Round-trip through JSON the way a stack file would be handed to the CLI.
rendered = json.dumps(cfn_template, indent=2, sort_keys=True)
parsed = json.loads(rendered)
props = parsed["Resources"]["DemoBucket"]["Properties"]
assert props["BucketName"] == STACK["bucket_name"]
assert props["VersioningConfiguration"]["Status"] == STACK["versioning"]
assert {t["Key"]: t["Value"] for t in props["Tags"]} == STACK["tags"]
print("template resources:", sorted(parsed["Resources"]))
print("CloudFormation describes the same bucket + versioning + tags")

## Step 3 — CDK (constructs that synthesize to a template)

CDK felt like the middle path: I write ordinary code with loops and conditionals, and it produces a template like the one above. No CDK library is installed in this environment, so I mirrored the construct shape with a tiny stub — the point is the layering (my code -> synthesized template -> deployed stack), not the library itself.

In [ ]:
try:
    import aws_cdk  # real CDK is not required for this comparison
    HAVE_CDK = True
except ImportError:
    HAVE_CDK = False

class BucketConstruct:
    """Stub standing in for a CDK Bucket construct: holds intent, synths to template."""
    def __init__(self, name, versioned, tags):
        self.name = name
        self.versioned = versioned
        self.tags = dict(tags)

    def synth(self):
        return {"AWS::S3::Bucket": {
            "BucketName": self.name,
            "VersioningConfiguration": {"Status": "Enabled" if self.versioned else "Suspended"},
            "Tags": [{"Key": k, "Value": v} for k, v in self.tags.items()],
        }}

construct = BucketConstruct(STACK["bucket_name"], versioned=True, tags=STACK["tags"])
synthed = construct.synth()
if not HAVE_CDK:
    print("aws-cdk-lib not installed — stub construct used to show the synth step")
bucket = synthed["AWS::S3::Bucket"]
assert bucket["BucketName"] == STACK["bucket_name"]
assert bucket["VersioningConfiguration"]["Status"] == STACK["versioning"]
assert {t["Key"]: t["Value"] for t in bucket["Tags"]} == STACK["tags"]
print("synthesized resource:", sorted(synthed))
print("CDK-style construct synths to the same bucket description")

## Verify — do all three agree, and when would I pick each?

Equivalence check first (the cells above already assert per-approach details; here I cross-check). Then the comparison as I see it after this exercise: boto3 when I need fine-grained control inside a larger program, CloudFormation when the team wants a reviewable template artifact with drift detection and rollback, CDK when the stack is big enough that template repetition hurts and the team is comfortable in a programming language.

In [ ]:
# Cross-approach equivalence: every path must name the same bucket, versioning, tags.
sdk_targets = {boto3_plan[0][1]["Bucket"]}
cfn_targets = {props["BucketName"]}
cdk_targets = {bucket["BucketName"]}
assert sdk_targets == cfn_targets == cdk_targets == {STACK["bucket_name"]}

rows = [
    ("boto3 SDK", "imperative calls", "caller orders + retries each call", "manual", "caller undoes"),
    ("CloudFormation", "declarative template", "service resolves dependencies", "drift detection", "automatic rollback"),
    ("CDK", "code -> synth -> template", "synth then service resolves", "drift detection", "automatic rollback"),
]
header = ("approach", "style", "ordering", "drift", "failure")
widths = [max(len(r[i]) for r in [header] + rows) for i in range(5)]
fmt = " | ".join("{:<%d}" % w for w in widths)
print(fmt.format(*header))
print("-+-|".join("-" * w for w in widths))
for r in rows:
    print(fmt.format(*r))
print("\nAll three describe:", sorted(cfn_targets), "+ versioning", STACK["versioning"], "+ tags", STACK["tags"])
print("VERIFY OK: same stack, three control styles")

## What I would try next

Two follow-ups: deploy the CloudFormation JSON for real in a sandbox account and run drift detection after tagging the bucket by hand, then re-express the CDK stub with the actual library to see how much synth output differs from my hand-written template. I also want to time how each path reports a failure (bad bucket name) — the SDK raises on the call, the template rolls back the stack.